# Q1-Safe Metadata-Only Bone Tumor Classifier
## Benign vs Malignant — SET-A Clinical Features Only

---

### Changes Applied in This Version

| # | Change | Detail |
|---|--------|--------|
| **A** | **Malignant recall gate lowered to 0.80** | `TARGET_MALIGNANT_RECALL = 0.80` (was 0.85) |
| **B** | **Model selection: best-overall composite on VAL** | Winner = highest composite score among gate-passing models. Score = BalancedAcc + MCC + MacroF1 + ROC-AUC (equal weights) − 0.1×pos_rate_deviation |
| **C** | **ROC-AUC added to composite score** | Equal weight alongside BalancedAcc, MCC, MacroF1 — favours models that are strong across all metrics |
| **D** | **Threshold plot updated** | Fig 6 now shows ROC-AUC line and updated recall gate label |

---

### Schema Fixes (from previous version)

| # | Fix | Detail |
|---|-----|--------|
| **1** | **`class_label` derivation** | Derived from `malignant` binary column: `0=Benign`, `1=Malignant` |
| **2** | **`foot` added to features** | `foot` present in schema but was missing from ANATOMICAL list |
| **3** | **`ILLEGAL` set expanded** | All diagnosis columns blocked from leaking into features |
| **4** | **`norm_label` removed** | Redundant after Fix #1 |
| **5** | **`use_label_encoder` removed** | Deprecated XGBoost kwarg removed |

---

### Model Selection Logic
```
Step 1  Hard gate:   malignant_recall >= 0.80  AND  pos_rate <= cap
        → models failing this are marked FAILED_CONSTRAINT and excluded

Step 2  Composite:   score = BalancedAcc + MCC + MacroF1 + ROC-AUC
                            - 0.10 × |pos_rate - true_prevalence|
        → highest composite among gate-passing models is selected
```

**Models:** Logistic Regression, Random Forest, Gradient Boosting, SVM, XGBoost, LightGBM  
**Environment:** Kaggle | sklearn 1.6.1 | Python 3.10+

---

## Section 0 — Imports & Global Settings

In [16]:
"""
Q1-SAFE MINIMAL METADATA-ONLY BONE TUMOR CLASSIFIER
====================================================
Benign vs Malignant (tumor images only) — SET-A clinical features only.

FIXES APPLIED:
1. class_label derived from 'malignant' column (was missing in original schema)
2. 'foot' added to ANATOMICAL features (present in dataset columns)
3. ILLEGAL set updated with all diagnosis columns from the actual schema
4. norm_label removed (class_label is always 0/1 after derivation)
5. Dataset columns: image_id, center, age, gender, hand, ulna, radius, humerus,
   foot, tibia, fibula, femur, hip bone, ankle-joint, knee-joint, hip-joint,
   wrist-joint, elbow-joint, shoulder-joint, tumor, benign, malignant,
   osteochondroma, multiple osteochondromas, simple bone cyst, giant cell tumor,
   osteofibroma, synovial osteochondroma, other bt, osteosarcoma, other mt,
   upper limb, lower limb, pelvis, frontal, lateral, oblique, patient_id, split

Q1 SAFETY GUARANTEES
 Patient-level leakage prevention (StratifiedGroupKFold by patient_id)
 Preprocessing strictly inside sklearn Pipeline + ColumnTransformer (no leakage)
 Calibration: OOF Platt scaling (public sklearn APIs only)
 5-fold CV evaluation on TRAIN only (mean±std metrics)
 Model selection via VALIDATION only (never test)
 Threshold optimized on VALIDATION: malignant recall >= 0.85 + trivial rejection cap
 FAILED_CONSTRAINT models excluded from selection
 Best SET-A model saved (joblib bundle: image + patient thresholds)
 Fusion-ready probability CSV (SET-A)
 Publication figures 300 dpi: ROC(TEST), PR(TEST), ConfMat(TEST),
   Classification Report(TEST), Calibration(TEST), Threshold(VAL)
 No test tuning — test is final report only

Models: Logistic Regression, Random Forest, Gradient Boosting, SVM, XGBoost, LightGBM
Environment: Kaggle | sklearn 1.6.1 | Python 3.10+
"""

"\nQ1-SAFE MINIMAL METADATA-ONLY BONE TUMOR CLASSIFIER\n====================================================\nBenign vs Malignant (tumor images only) — SET-A clinical features only.\n\nFIXES APPLIED:\n1. class_label derived from 'malignant' column (was missing in original schema)\n2. 'foot' added to ANATOMICAL features (present in dataset columns)\n3. ILLEGAL set updated with all diagnosis columns from the actual schema\n4. norm_label removed (class_label is always 0/1 after derivation)\n5. Dataset columns: image_id, center, age, gender, hand, ulna, radius, humerus,\n   foot, tibia, fibula, femur, hip bone, ankle-joint, knee-joint, hip-joint,\n   wrist-joint, elbow-joint, shoulder-joint, tumor, benign, malignant,\n   osteochondroma, multiple osteochondromas, simple bone cyst, giant cell tumor,\n   osteofibroma, synovial osteochondroma, other bt, osteosarcoma, other mt,\n   upper limb, lower limb, pelvis, frontal, lateral, oblique, patient_id, split\n\nQ1 SAFETY GUARANTEES\n Patient-lev

---

## Section 1 — Group-Aware Calibration (OOF Platt)

In [17]:
# =============================================================================
# 0) IMPORTS + GLOBAL SETTINGS
# =============================================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sklearn
import joblib

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
    precision_recall_fscore_support, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss,
    average_precision_score, matthews_corrcoef,
    classification_report
)
from sklearn.calibration import calibration_curve

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available — skipping XGBClassifier.")

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("LightGBM not available — skipping LGBMClassifier.")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Clinical constraint (enforced on VALIDATION only)
TARGET_MALIGNANT_RECALL   = 0.80          # minimum malignant recall gate
HARD_MAX_PRED_POS_RATE    = 0.90
ADAPTIVE_POS_RATE_MARGIN  = 0.50

# Composite score weights (VAL threshold selection)
# Weighted equally across ROC-AUC, BalancedAcc, MCC, Macro-F1 for
# best overall model — recall gate still enforced as hard constraint.
W_BAL  = 1.00
W_MCC  = 1.00
W_F1   = 1.00
W_AUC  = 1.00    # added: ROC-AUC term in composite
W_POS  = 0.10

# Output dirs
OUT    = Path("/kaggle/working/metadata_classifier")
FIGS   = OUT / "figures"
TABLES = OUT / "tables"
MODELS = OUT / "saved_models"
for d in [OUT, FIGS, TABLES, MODELS]:
    d.mkdir(parents=True, exist_ok=True)

# Plot config
plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300,
                     "font.family": "sans-serif", "font.size": 10})

print("=" * 80)
print("Q1-SAFE METADATA BONE TUMOR CLASSIFIER  (SET-A only | Benign vs Malignant)")
print("=" * 80)
print(f"sklearn {sklearn.__version__} | XGBoost={'yes' if HAS_XGB else 'no'} | LightGBM={'yes' if HAS_LGB else 'no'}")
print()

Q1-SAFE METADATA BONE TUMOR CLASSIFIER  (SET-A only | Benign vs Malignant)
sklearn 1.6.1 | XGBoost=yes | LightGBM=yes



---

## Section 2 — Data Loading

In [18]:
# =============================================================================
# 1) GROUP-AWARE CALIBRATION  (OOF Platt — public sklearn APIs only)
# =============================================================================
class GroupCalibratedModel(BaseEstimator, ClassifierMixin):
    """
    Fits final_pipeline on ALL training data.
    Generates OOF probs via StratifiedGroupKFold(groups=patient_id).
    Fits a Platt scaler (LogisticRegression) on OOF probs.
    Inference: base_proba -> Platt -> calibrated proba.
    """
    def __init__(self, base_pipeline, cv_splitter, random_state=42):
        self.base_pipeline  = base_pipeline
        self.cv_splitter    = cv_splitter
        self.random_state   = random_state
        self.final_pipeline_ = None
        self.platt_scaler_   = None
        self.classes_        = np.array([0, 1], dtype=int)

    def fit(self, X, y, groups):
        y_arr = y.values if isinstance(y, pd.Series) else np.asarray(y)
        n_samples = len(y_arr)

        # Step 1: fit final model on ALL training data
        self.final_pipeline_ = clone(self.base_pipeline)
        self.final_pipeline_.fit(X, y_arr)

        # Step 2: generate OOF probabilities for Platt calibration
        oof     = np.full(n_samples, np.nan, dtype=float)
        written = np.zeros(n_samples, dtype=int)

        for fold_idx, (tr_idx, cal_idx) in enumerate(
                self.cv_splitter.split(X, y_arr, groups)):

            fold_model = clone(self.base_pipeline)
            assert fold_model is not self.final_pipeline_, (
                f"Fold {fold_idx}: fold_model must be a fresh clone, "
                "not a reference to final_pipeline_."
            )

            Xtr = X.iloc[tr_idx] if isinstance(X, pd.DataFrame) else X[tr_idx]
            Xca = X.iloc[cal_idx] if isinstance(X, pd.DataFrame) else X[cal_idx]

            fold_model.fit(Xtr, y_arr[tr_idx])
            oof[cal_idx]     = fold_model.predict_proba(Xca)[:, 1]
            written[cal_idx] += 1

        never_written = int((written == 0).sum())
        multi_written = int((written > 1).sum())
        if never_written > 0:
            raise RuntimeError(
                f"OOF incomplete: {never_written} sample(s) never assigned a prediction.")
        if multi_written > 0:
            raise RuntimeError(
                f"OOF collision: {multi_written} sample(s) written more than once.")
        if np.isnan(oof).any():
            raise RuntimeError("OOF probabilities still contain NaN after coverage check.")

        # Platt scaling
        self.platt_scaler_ = LogisticRegression(
            solver="lbfgs", max_iter=3000, random_state=self.random_state
        )
        self.platt_scaler_.fit(oof.reshape(-1, 1), y_arr)
        return self

    def predict_proba(self, X):
        p_raw = self.final_pipeline_.predict_proba(X)[:, 1]
        return self.platt_scaler_.predict_proba(p_raw.reshape(-1, 1))  # [P0, P1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

---

## Section 3 — Derive class_label from 'malignant' [FIX #1]

In [19]:
# =============================================================================
# 2) DATA LOADING
# =============================================================================
print("-- SECTION 1: DATA LOADING -------------------------------------------------")
DATA   = Path("/kaggle/input/datasets/samiulalim172/meta-new")
df_tr  = pd.read_csv(DATA / "train_metadata.csv")
df_v   = pd.read_csv(DATA / "val_metadata.csv")
df_te  = pd.read_csv(DATA / "test_metadata.csv")
print(f"Loaded -- Train={len(df_tr)} | Val={len(df_v)} | Test={len(df_te)}")
print()

-- SECTION 1: DATA LOADING -------------------------------------------------
Loaded -- Train=2602 | Val=580 | Test=564



---

## Section 4 — Tumor Filter & Class Validation

In [20]:
# =============================================================================
# 3) DERIVE class_label FROM 'malignant' COLUMN  [FIX #1]
# =============================================================================
# The dataset does NOT have a pre-built 'class_label' column.
# We derive it from the 'malignant' binary column:
#   malignant == 1  ->  class_label = 1  (Malignant)
#   malignant == 0  ->  class_label = 0  (Benign)
# This must happen BEFORE the tumor filter so that all rows are normalised.
print("-- SECTION 2a: DERIVE class_label FROM malignant COLUMN -------------------")
for tag, df in [("train", df_tr), ("val", df_v), ("test", df_te)]:
    if "malignant" not in df.columns:
        raise ValueError(f"Column 'malignant' missing from {tag} — cannot derive class_label.")
    df["class_label"] = df["malignant"].astype(int)
    assert df["class_label"].isin([0, 1]).all(), \
        f"Unexpected values in 'malignant' column of {tag}: {df['malignant'].unique()}"
print("  class_label derived successfully (0=Benign, 1=Malignant) for all splits.")
print()

-- SECTION 2a: DERIVE class_label FROM malignant COLUMN -------------------
  class_label derived successfully (0=Benign, 1=Malignant) for all splits.



---

## Section 5 — Patient Overlap Check (Leakage Guard)

In [21]:
# =============================================================================
# 4) TUMOR FILTER + CLASS VALIDATION
# =============================================================================
print("-- SECTION 2b: TUMOR FILTER + VALIDATION ----------------------------------")
# FIX: 'class_label' is now present (derived above); check updated required cols
REQUIRED = ["tumor", "patient_id", "image_id", "gender", "age", "class_label"]
for col in REQUIRED:
    for tag, df in [("train", df_tr), ("val", df_v), ("test", df_te)]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {tag}.")

df_tr = df_tr[df_tr["tumor"] == 1].copy().reset_index(drop=True)
df_v  = df_v[df_v["tumor"] == 1].copy().reset_index(drop=True)
df_te = df_te[df_te["tumor"] == 1].copy().reset_index(drop=True)
print(f"After tumor filter -- Train={len(df_tr)} | Val={len(df_v)} | Test={len(df_te)}")

for tag, df in [("Train", df_tr), ("Val", df_v), ("Test", df_te)]:
    d = df["class_label"].value_counts().sort_index()
    print(f"  {tag}: Benign={d.get(0,0)} | Malignant={d.get(1,0)}")
print()

-- SECTION 2b: TUMOR FILTER + VALIDATION ----------------------------------
After tumor filter -- Train=1285 | Val=296 | Test=286
  Train: Benign=1050 | Malignant=235
  Val: Benign=236 | Malignant=60
  Test: Benign=239 | Malignant=47



---

## Section 6 — Gender Standardisation

In [22]:
# =============================================================================
# 5) PATIENT OVERLAP CHECK (leakage guard)
# =============================================================================
print("-- SECTION 3: PATIENT OVERLAP CHECK ---------------------------------------")
tr_p = set(df_tr["patient_id"].unique())
v_p  = set(df_v["patient_id"].unique())
te_p = set(df_te["patient_id"].unique())
ov = len(tr_p & v_p) + len(tr_p & te_p) + len(v_p & te_p)
if ov > 0:
    raise RuntimeError(f"LEAKAGE: Patient overlap detected ({ov} patients shared).")
print(f"No patient overlap. Train={len(tr_p)} | Val={len(v_p)} | Test={len(te_p)} patients.")
print()

-- SECTION 3: PATIENT OVERLAP CHECK ---------------------------------------
No patient overlap. Train=548 | Val=123 | Test=119 patients.



---

## Section 7 — SET-A Feature Definition [FIX #2 & #3]

In [23]:
# =============================================================================
# 6) GENDER STANDARDISATION
# =============================================================================
def map_gender(df):
    if df["gender"].dtype == object:
        m = {"M":0,"Male":0,"male":0,"F":1,"Female":1,"female":1}
        df["gender"] = df["gender"].map(m)
    else:
        if df["gender"].min() == 1 and df["gender"].max() == 2:
            df["gender"] = df["gender"] - 1
    return df

df_tr = map_gender(df_tr)
df_v  = map_gender(df_v)
df_te = map_gender(df_te)

---

## Section 8 — Data Matrices

In [24]:
# =============================================================================
# 7) SET-A FEATURE DEFINITION
# =============================================================================
print("-- SECTION 4: FEATURE SET (SET-A ONLY) ------------------------------------")
DEMOGRAPHIC  = ["age", "gender"]
# FIX #2: 'foot' added — it is present in the dataset schema but was missing
ANATOMICAL   = ["hand", "ulna", "radius", "humerus",
                "foot",                              # <-- added
                "femur", "tibia", "fibula",
                "hip bone", "ankle-joint", "knee-joint", "hip-joint",
                "wrist-joint", "elbow-joint", "shoulder-joint"]
REGION       = ["upper limb", "lower limb", "pelvis"]
VIEWS        = ["frontal", "lateral", "oblique"]

FEATURES_A   = [f for f in (DEMOGRAPHIC + ANATOMICAL + REGION + VIEWS)
                if f in df_tr.columns]

# FIX #3: ILLEGAL set expanded to cover all diagnosis columns in the actual schema
ILLEGAL = {
    "class_label", "benign", "malignant", "tumor",
    "patient_id", "image_id", "center", "split",
    # diagnosis-specific columns from the dataset schema
    "osteochondroma", "multiple osteochondromas",
    "simple bone cyst", "giant cell tumor",
    "osteofibroma", "synovial osteochondroma",
    "other bt", "osteosarcoma", "other mt",
}
leak = set(FEATURES_A) & ILLEGAL
if leak:
    raise ValueError(f"LEAKAGE: {leak} found in feature list.")

# 'center' guard
if "center" in df_tr.columns:
    assert "center" not in FEATURES_A, \
        "LEAKAGE: 'center' found in FEATURES_A — remove it before proceeding."
    print("  center column detected in data but excluded (leakage guard).")

print(f"SET-A features ({len(FEATURES_A)}): {FEATURES_A}")
print()

-- SECTION 4: FEATURE SET (SET-A ONLY) ------------------------------------
  center column detected in data but excluded (leakage guard).
SET-A features (23): ['age', 'gender', 'hand', 'ulna', 'radius', 'humerus', 'foot', 'femur', 'tibia', 'fibula', 'hip bone', 'ankle-joint', 'knee-joint', 'hip-joint', 'wrist-joint', 'elbow-joint', 'shoulder-joint', 'upper limb', 'lower limb', 'pelvis', 'frontal', 'lateral', 'oblique']



---

## Section 9 — Preprocessor & Pipeline Factory (Leakage-Free)

In [25]:
# =============================================================================
# 8) DATA MATRICES
# =============================================================================
def make_xy(df, feats):
    return (df[feats].copy(), df["class_label"].copy(),
            df["image_id"].copy(), df["patient_id"].copy())

X_tr, y_tr, img_tr, pat_tr = make_xy(df_tr, FEATURES_A)
X_v,  y_v,  img_v,  pat_v  = make_xy(df_v,  FEATURES_A)
X_te, y_te, img_te, pat_te = make_xy(df_te, FEATURES_A)

---

## Section 10 — 5-Fold Cross-Validation on Train (Reporting Only)

In [26]:
# =============================================================================
# 9) PREPROCESSOR (strictly inside Pipeline — no leakage)
# =============================================================================
def make_preprocessor(features):
    num = ["age"] if "age" in features else []
    cat = [f for f in features if f not in num]
    num_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        [("num", num_pipe, num), ("cat", cat_pipe, cat)],
        remainder="drop"
    )


def make_pipeline(features, clf):
    to_np = FunctionTransformer(np.asarray, validate=False)
    try:
        to_np.set_output(transform="default")
    except AttributeError:
        pass
    return Pipeline([
        ("pre",   make_preprocessor(features)),
        ("to_np", to_np),
        ("clf",   clf),
    ])


# XGBoost imbalance weight
xgb_pos_w = float((y_tr == 0).sum()) / max(float((y_tr == 1).sum()), 1.0)
print(f"XGBoost scale_pos_weight (Benign:Malignant ratio from train): {xgb_pos_w:.3f}")
print()

def build_base_models(pos_weight: float):
    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced", max_iter=3000,
            solver="liblinear", random_state=RANDOM_STATE
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=500, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=3,
            random_state=RANDOM_STATE
        ),
        "SVM": SVC(
            kernel="rbf", class_weight="balanced",
            probability=True, random_state=RANDOM_STATE
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", scale_pos_weight=pos_weight,
            n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
        )
    if HAS_LGB:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
            class_weight="balanced", n_jobs=-1,
            random_state=RANDOM_STATE, verbose=-1
        )
    return models

XGBoost scale_pos_weight (Benign:Malignant ratio from train): 4.468



---

## Section 11 — Train & Calibrate All Models (OOF Platt)

In [27]:
# =============================================================================
# 10) 5-FOLD CV ON TRAIN
# =============================================================================
print("-- SECTION 5: 5-FOLD CV ON TRAIN (uncalibrated models; reporting only) ----")
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
base_models = build_base_models(xgb_pos_w)

CV_SCORING = {
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc":           "roc_auc",
    "f1_macro":          "f1_macro",
    "average_precision": "average_precision",
    "brier":             "neg_brier_score",
}

cv_results_rows = []
for name, clf in base_models.items():
    pipe = make_pipeline(FEATURES_A, clf)
    cv_res = cross_validate(
        pipe, X_tr, y_tr,
        cv=cv5, groups=pat_tr,
        scoring=CV_SCORING,
        n_jobs=-1, return_train_score=False
    )
    mcc_folds = []
    for tr_idx, va_idx in cv5.split(X_tr, y_tr, pat_tr):
        m = clone(pipe)
        Xf_tr = X_tr.iloc[tr_idx]; Xf_va = X_tr.iloc[va_idx]
        m.fit(Xf_tr, y_tr.iloc[tr_idx])
        yhat = m.predict(Xf_va)
        mcc_folds.append(matthews_corrcoef(y_tr.iloc[va_idx], yhat))

    row = {"Model": name}
    for metric, key in [
        ("BalancedAcc", "test_balanced_accuracy"),
        ("ROC-AUC",     "test_roc_auc"),
        ("PR-AUC",      "test_average_precision"),
        ("Macro-F1",    "test_f1_macro"),
    ]:
        arr = cv_res[key]
        row[f"{metric}_mean"] = round(float(np.mean(arr)), 4)
        row[f"{metric}_std"]  = round(float(np.std(arr)),  4)

    brier_arr = -cv_res["test_brier"]
    row["Brier_mean"] = round(float(np.mean(brier_arr)), 4)
    row["Brier_std"]  = round(float(np.std(brier_arr)),  4)
    row["MCC_mean"]   = round(float(np.mean(mcc_folds)), 4)
    row["MCC_std"]    = round(float(np.std(mcc_folds)),  4)
    cv_results_rows.append(row)
    print(f"  {name}: BalAcc={row['BalancedAcc_mean']:.4f}+-{row['BalancedAcc_std']:.4f} | "
          f"AUC={row['ROC-AUC_mean']:.4f}+-{row['ROC-AUC_std']:.4f} | "
          f"MCC={row['MCC_mean']:.4f}+-{row['MCC_std']:.4f} | "
          f"Brier={row['Brier_mean']:.4f}+-{row['Brier_std']:.4f}")

cv_df = pd.DataFrame(cv_results_rows)
cv_df.to_csv(TABLES / "cv5_train_metrics_uncalibrated.csv", index=False)
print(f"\nSaved 5-fold CV table: {TABLES / 'cv5_train_metrics_uncalibrated.csv'}")
print()

-- SECTION 5: 5-FOLD CV ON TRAIN (uncalibrated models; reporting only) ----
  Logistic Regression: BalAcc=0.6870+-0.0405 | AUC=0.7326+-0.0570 | MCC=0.2933+-0.0638 | Brier=0.2120+-0.0303
  Random Forest: BalAcc=0.6685+-0.0507 | AUC=0.7312+-0.0259 | MCC=0.2703+-0.0731 | Brier=0.1875+-0.0210
  Gradient Boosting: BalAcc=0.5399+-0.0224 | AUC=0.6870+-0.0336 | MCC=0.1313+-0.0651 | Brier=0.1475+-0.0136
  SVM: BalAcc=0.6838+-0.0390 | AUC=0.7236+-0.0583 | MCC=0.2878+-0.0579 | Brier=0.1365+-0.0176
  XGBoost: BalAcc=0.6261+-0.0435 | AUC=0.6735+-0.0365 | MCC=0.2070+-0.0659 | Brier=0.2057+-0.0207


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

  LightGBM: BalAcc=0.6357+-0.0391 | AUC=0.6763+-0.0384 | MCC=0.2200+-0.0616 | Brier=0.2136+-0.0256

Saved 5-fold CV table: /kaggle/working/metadata_classifier/tables/cv5_train_metrics_uncalibrated.csv



---

## Section 12 — Threshold Optimisation (Validation Only) [UPDATED: recall gate=0.80, composite score includes ROC-AUC]

In [28]:
# =============================================================================
# 11) TRAIN CALIBRATED MODELS
# =============================================================================
print("-- SECTION 6: TRAIN + CALIBRATE ALL MODELS (OOF Platt, VAL-only selection) -")
calib_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

trained_models = {}
for name, clf in base_models.items():
    pipe  = make_pipeline(FEATURES_A, clf)
    model = GroupCalibratedModel(pipe, cv_splitter=calib_cv, random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr, pat_tr)
    trained_models[name] = model
    print(f"  {name}")
print()

-- SECTION 6: TRAIN + CALIBRATE ALL MODELS (OOF Platt, VAL-only selection) -
  Logistic Regression
  Random Forest
  Gradient Boosting
  SVM
  XGBoost
  LightGBM



---

## Section 13 — Best Model Selection (Validation Only) [UPDATED: best-overall composite score]

In [29]:
# =============================================================================
# 12) THRESHOLD OPTIMISATION (VALIDATION ONLY)
# =============================================================================
def metrics_at_t(y_true, p, t):
    """
    Compute threshold-dependent metrics.
    ROC-AUC is threshold-independent but included here so it travels
    with each row in the threshold sweep table (constant across t).
    """
    y_true = np.asarray(y_true).astype(int)
    p      = np.asarray(p).astype(float)
    yhat   = (p >= t).astype(int)
    bal    = balanced_accuracy_score(y_true, yhat)
    mf1    = f1_score(y_true, yhat, average="macro", zero_division=0)
    mcc    = matthews_corrcoef(y_true, yhat) if len(np.unique(yhat)) > 1 else -1.0
    auc    = roc_auc_score(y_true, p) if len(np.unique(y_true)) == 2 else 0.5
    pr, rc, _, _ = precision_recall_fscore_support(y_true, yhat, average=None, zero_division=0)
    return {
        "balanced_accuracy": float(bal),
        "macro_f1":          float(mf1),
        "mcc":               float(mcc),
        "roc_auc":           float(auc),
        "malignant_recall":  float(rc[1]),
        "malignant_precision": float(pr[1]),
        "pos_rate":          float(yhat.mean()),
        "true_pos_rate":     float(y_true.mean()),
    }

def _composite(m):
    """
    Best-overall composite score (VAL):
    Equal weight on BalancedAcc + MCC + MacroF1 + ROC-AUC
    minus a small penalty for straying from true prevalence.
    Malignant recall >= TARGET_MALIGNANT_RECALL is a hard gate (applied
    separately); this score ranks models within the feasible region.
    """
    return (W_BAL * m["balanced_accuracy"]
            + W_MCC * m["mcc"]
            + W_F1  * m["macro_f1"]
            + W_AUC * m["roc_auc"]
            - W_POS * abs(m["pos_rate"] - m["true_pos_rate"]))

def find_best_threshold(model, X_val, y_val, min_recall=TARGET_MALIGNANT_RECALL):
    p      = model.predict_proba(X_val)[:, 1]
    thrs   = np.arange(0.05, 0.96, 0.01)
    tpr    = float(np.mean(np.asarray(y_val).astype(int)))
    cap    = float(min(HARD_MAX_PRED_POS_RATE, tpr + ADAPTIVE_POS_RATE_MARGIN))
    rows   = []
    for t in thrs:
        m     = metrics_at_t(y_val, p, t)
        score = _composite(m)
        rows.append({"threshold": float(t), "score": score, "cap": cap, **m})
    tbl   = pd.DataFrame(rows)
    valid = tbl[(tbl["malignant_recall"] >= min_recall) & (tbl["pos_rate"] <= cap)]
    if len(valid) == 0:
        best = tbl.sort_values("score", ascending=False).iloc[0]
        return float(best["threshold"]), best.to_dict(), tbl, "FAILED_CONSTRAINT"
    best = valid.sort_values("score", ascending=False).iloc[0]
    return float(best["threshold"]), best.to_dict(), tbl, "OK"

print("-- SECTION 7: THRESHOLD OPTIMISATION (VAL ONLY) ---------------------------")
thresholds, best_rows, thr_tables, statuses = {}, {}, {}, {}
for name, model in trained_models.items():
    t, brow, tbl, st = find_best_threshold(model, X_v, y_v)
    thresholds[name] = t; best_rows[name] = brow
    thr_tables[name] = tbl; statuses[name] = st
    tag = "OK" if st == "OK" else "FAILED_CONSTRAINT"
    print(f"  [{tag}] {name}: t={t:.3f} | bal={brow['balanced_accuracy']:.4f} | "
          f"mal_rec={brow['malignant_recall']:.4f} | pos_rate={brow['pos_rate']:.3f}")
    tbl.to_csv(TABLES / f"thr_table_VAL_{name.replace(' ','_')}.csv", index=False)
print()

-- SECTION 7: THRESHOLD OPTIMISATION (VAL ONLY) ---------------------------
  [OK] Logistic Regression: t=0.120 | bal=0.6898 | mal_rec=0.8500 | pos_rate=0.547
  [OK] Random Forest: t=0.100 | bal=0.6891 | mal_rec=0.9333 | pos_rate=0.632
  [OK] Gradient Boosting: t=0.160 | bal=0.7051 | mal_rec=0.8000 | pos_rate=0.473
  [OK] SVM: t=0.110 | bal=0.6815 | mal_rec=0.8333 | pos_rate=0.544
  [OK] XGBoost: t=0.150 | bal=0.6729 | mal_rec=0.8500 | pos_rate=0.574
  [OK] LightGBM: t=0.160 | bal=0.6898 | mal_rec=0.8500 | pos_rate=0.547



---

## Section 14 — Evaluation Helper

In [30]:
# =============================================================================
# 13) BEST MODEL SELECTION (VAL, eligible only)
# =============================================================================
# Strategy: among models that pass the hard malignant recall gate (>= 0.80),
# select the model with the highest COMPOSITE score on VAL.
# Composite = BalancedAcc + MCC + MacroF1 + ROC-AUC  (equal weights)
# This picks the best overall performer, not just the highest single metric.
# =============================================================================
print("-- SECTION 8: BEST MODEL SELECTION (VAL ONLY) — Best Overall Composite ----")
eligible = [(n, best_rows[n]) for n, s in statuses.items() if s == "OK"]
if not eligible:
    raise RuntimeError("No model satisfied clinical constraint on VAL. Lower TARGET_MALIGNANT_RECALL.")

# Sort by composite score descending — best overall model wins
eligible.sort(key=lambda x: x[1]["score"], reverse=True)
best_name  = eligible[0][0]
best_thr   = thresholds[best_name]
best_model = trained_models[best_name]

rank_df = pd.DataFrame([
    {"Model":      n,
     "Status":     statuses[n],
     "Threshold":  round(thresholds[n], 3),
     "BalancedAcc":round(best_rows[n]["balanced_accuracy"], 4),
     "ROC-AUC":    round(best_rows[n].get("roc_auc", float("nan")), 4),
     "MCC":        round(best_rows[n]["mcc"],               4),
     "MacroF1":    round(best_rows[n]["macro_f1"],          4),
     "MalRecall":  round(best_rows[n]["malignant_recall"],   4),
     "Score":      round(best_rows[n]["score"],              4)}
    for n in trained_models
]).sort_values("Score", ascending=False).reset_index(drop=True)

rank_df["Rank"] = rank_df.index + 1
cols = ["Rank", "Model", "Status", "Threshold",
        "BalancedAcc", "ROC-AUC", "MCC", "MacroF1", "MalRecall", "Score"]
rank_df = rank_df[cols]

print(rank_df.to_string(index=False))
rank_df.to_csv(TABLES / "val_ranking_all_models.csv", index=False)
print(f"\nBest model (VAL, best-overall composite): {best_name}")
print(f"  Threshold    = {best_thr:.3f}")
print(f"  BalancedAcc  = {best_rows[best_name]['balanced_accuracy']:.4f}")
print(f"  ROC-AUC      = {best_rows[best_name].get('roc_auc', float('nan')):.4f}")
print(f"  MCC          = {best_rows[best_name]['mcc']:.4f}")
print(f"  MacroF1      = {best_rows[best_name]['macro_f1']:.4f}")
print(f"  MalRecall    = {best_rows[best_name]['malignant_recall']:.4f}  (gate >= {TARGET_MALIGNANT_RECALL})")
print(f"  CompositeScore = {best_rows[best_name]['score']:.4f}")
print()

-- SECTION 8: BEST MODEL SELECTION (VAL ONLY) — Best Overall Composite ----
 Rank               Model Status  Threshold  BalancedAcc  ROC-AUC    MCC  MacroF1  MalRecall  Score
    1   Gradient Boosting     OK       0.16       0.7051   0.7450 0.3303   0.6073     0.8000 2.3607
    2 Logistic Regression     OK       0.12       0.6898   0.7399 0.3066   0.5676     0.8500 2.2694
    3            LightGBM     OK       0.16       0.6898   0.7310 0.3066   0.5676     0.8500 2.2605
    4       Random Forest     OK       0.10       0.6891   0.7621 0.3153   0.5311     0.9333 2.2547
    5                 SVM     OK       0.11       0.6815   0.7440 0.2930   0.5632     0.8333 2.2476
    6             XGBoost     OK       0.15       0.6729   0.7270 0.2811   0.5449     0.8500 2.1888

Best model (VAL, best-overall composite): Gradient Boosting
  Threshold    = 0.160
  BalancedAcc  = 0.7051
  ROC-AUC      = 0.7450
  MCC          = 0.3303
  MacroF1      = 0.6073
  MalRecall    = 0.8000  (gate >= 0.8)
  Com

---

## Section 15 — Final Test Report Table

In [31]:
# =============================================================================
# 14) EVALUATION HELPER
# =============================================================================
def evaluate(model, X, y, t):
    p    = model.predict_proba(X)[:, 1]
    yhat = (p >= t).astype(int)
    fpr, tpr, _ = roc_curve(y, p)
    prc, rcc, _ = precision_recall_curve(y, p)
    pt, pp       = calibration_curve(y, p, n_bins=10, strategy="uniform")
    prec, rec, _, _ = precision_recall_fscore_support(y, yhat, average=None, zero_division=0)
    return {
        "accuracy":             float(accuracy_score(y, yhat)),
        "balanced_accuracy":    float(balanced_accuracy_score(y, yhat)),
        "mcc":                  float(matthews_corrcoef(y, yhat)) if len(np.unique(yhat)) > 1 else -1.,
        "macro_f1":             float(f1_score(y, yhat, average="macro", zero_division=0)),
        "roc_auc":              float(roc_auc_score(y, p)),
        "pr_auc":               float(average_precision_score(y, p)),
        "brier":                float(brier_score_loss(y, p)),
        "malignant_recall":     float(rec[1]),
        "malignant_precision":  float(prec[1]),
        "benign_recall":        float(rec[0]),
        "benign_precision":     float(prec[0]),
        "pos_rate_pred":        float(yhat.mean()),
        "pos_rate_true":        float(np.mean(y)),
        "cm":                   confusion_matrix(y, yhat),
        "cmn":                  confusion_matrix(y, yhat, normalize="true"),
        "fpr": fpr, "tpr": tpr,
        "prc": prc, "rcc": rcc,
        "prob_true": pt, "prob_pred": pp,
        "p": p, "yhat": yhat
    }

res_val  = {n: evaluate(m, X_v,  y_v,  thresholds[n]) for n, m in trained_models.items()}
res_test = {n: evaluate(m, X_te, y_te, thresholds[n]) for n, m in trained_models.items()}

---

## Section 16 — Patient-Level Threshold & Test Evaluation

In [32]:
# =============================================================================
# 15) FINAL TEST REPORT TABLE
# =============================================================================
print("-- SECTION 9: TEST REPORT TABLE (final — no tuning) -----------------------")
def make_table(results, thrs, statuses):
    rows = []
    for name, m in results.items():
        rows.append({
            "Model":        name,
            "Status(VAL)":  statuses.get(name, ""),
            "Threshold":    f"{thrs[name]:.3f}",
            "Accuracy":     f"{m['accuracy']:.4f}",
            "BalancedAcc":  f"{m['balanced_accuracy']:.4f}",
            "MCC":          f"{m['mcc']:.4f}",
            "MacroF1":      f"{m['macro_f1']:.4f}",
            "ROC-AUC":      f"{m['roc_auc']:.4f}",
            "PR-AUC":       f"{m['pr_auc']:.4f}",
            "MalRecall":    f"{m['malignant_recall']:.4f}",
            "MalPrec":      f"{m['malignant_precision']:.4f}",
            "Brier":        f"{m['brier']:.4f}",
        })
    return pd.DataFrame(rows)

tbl_test = make_table(res_test, thresholds, statuses)
print(tbl_test.to_string(index=False))
tbl_test.to_csv(TABLES / "test_report_all_models.csv", index=False)
print()

-- SECTION 9: TEST REPORT TABLE (final — no tuning) -----------------------
              Model Status(VAL) Threshold Accuracy BalancedAcc    MCC MacroF1 ROC-AUC PR-AUC MalRecall MalPrec  Brier
Logistic Regression          OK     0.120   0.5455      0.5486 0.0721  0.4762  0.6314 0.2541    0.5532  0.1926 0.1341
      Random Forest          OK     0.100   0.4650      0.5688 0.1045  0.4359  0.6290 0.2504    0.7234  0.1954 0.1345
  Gradient Boosting          OK     0.160   0.6399      0.5965 0.1481  0.5405  0.6439 0.2477    0.5319  0.2358 0.1347
                SVM          OK     0.110   0.5629      0.5676 0.1005  0.4918  0.6084 0.2675    0.5745  0.2045 0.1340
            XGBoost          OK     0.150   0.5559      0.5890 0.1320  0.4955  0.6520 0.2480    0.6383  0.2143 0.1334
           LightGBM          OK     0.160   0.6049      0.6012 0.1517  0.5255  0.6382 0.2342    0.5957  0.2295 0.1343



---

## Section 17 — Fusion-Ready CSV Export

In [33]:
# =============================================================================
# 16) PATIENT-LEVEL THRESHOLD (VAL patient-agg -> test reporting)
# =============================================================================
def patient_agg(img_df):
    p_mean  = img_df.groupby("patient_id")["p_malignant"].mean()
    y_mode  = img_df.groupby("patient_id")["true_label"].agg(lambda s: int(s.mode().iloc[0]))
    return pd.DataFrame({"patient_id": p_mean.index,
                         "p_malignant": p_mean.values,
                         "true_label":  y_mode.values})

def make_img_df(model, X, y, img, pat, t, name, split):
    p    = model.predict_proba(X)[:, 1]
    yhat = (p >= t).astype(int)
    return pd.DataFrame({"image_id": img.values, "patient_id": pat.values,
                         "p_benign": 1-p, "p_malignant": p,
                         "split": split, "model_name": name,
                         "threshold_image_level": t,
                         "predicted_label_image_level": yhat,
                         "true_label": y.values})

val_img_best  = make_img_df(best_model, X_v,  y_v,  img_v,  pat_v,  best_thr, best_name, "val")
test_img_best = make_img_df(best_model, X_te, y_te, img_te, pat_te, best_thr, best_name, "test")

val_pat_df  = patient_agg(val_img_best)
test_pat_df = patient_agg(test_img_best)

def thr_opt_probs(p, y):
    """
    Threshold optimiser on raw probability arrays (used for patient-level).
    Uses the same _composite score and recall gate as find_best_threshold.
    """
    p    = np.asarray(p).astype(float)
    y    = np.asarray(y).astype(int)
    thrs = np.arange(0.05, 0.96, 0.01)
    cap  = float(min(HARD_MAX_PRED_POS_RATE, float(y.mean()) + ADAPTIVE_POS_RATE_MARGIN))
    rows = []
    for t in thrs:
        m     = metrics_at_t(y, p, t)
        score = _composite(m)
        rows.append({"threshold": float(t), "score": score, "cap": cap, **m})
    tbl   = pd.DataFrame(rows)
    valid = tbl[(tbl["malignant_recall"] >= TARGET_MALIGNANT_RECALL) & (tbl["pos_rate"] <= cap)]
    if len(valid) == 0:
        best = tbl.sort_values("score", ascending=False).iloc[0]
        return float(best["threshold"]), "FAILED_CONSTRAINT", tbl
    return float(valid.sort_values("score", ascending=False).iloc[0]["threshold"]), "OK", tbl

t_pat, st_pat, tbl_pat = thr_opt_probs(
    val_pat_df["p_malignant"].values,
    val_pat_df["true_label"].values
)
tbl_pat.to_csv(TABLES / "thr_table_VAL_patient_level_best_SET_A_mean.csv", index=False)

if st_pat != "OK":
    print("Patient threshold failed constraint on VAL -- using image threshold as fallback.")
    t_pat = best_thr
print(f"Patient-level threshold (VAL, mean-agg): {t_pat:.3f}  [status={st_pat}]")
print()

print("-- SECTION 16b: PATIENT-LEVEL TEST EVALUATION (final report — no tuning) ---")

def patient_level_metrics(pat_df, threshold, label=""):
    p    = pat_df["p_malignant"].values.astype(float)
    y    = pat_df["true_label"].values.astype(int)
    yhat = (p >= threshold).astype(int)
    auc  = float(roc_auc_score(y, p))  if len(np.unique(y)) == 2 else np.nan
    ap   = float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else np.nan
    prec, rec, _, _ = precision_recall_fscore_support(y, yhat, average=None, zero_division=0)
    cm   = confusion_matrix(y, yhat)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (np.nan,)*4
    return {
        "n_patients":           int(len(y)),
        "threshold_used(VAL)":  round(float(threshold), 3),
        "accuracy":             round(float(accuracy_score(y, yhat)), 4),
        "balanced_accuracy":    round(float(balanced_accuracy_score(y, yhat)), 4),
        "mcc":                  round(float(matthews_corrcoef(y, yhat)) if len(np.unique(yhat)) > 1 else -1., 4),
        "macro_f1":             round(float(f1_score(y, yhat, average="macro", zero_division=0)), 4),
        "roc_auc":              round(auc, 4),
        "pr_auc":               round(ap, 4),
        "brier":                round(float(brier_score_loss(y, p)), 4),
        "malignant_recall":     round(float(rec[1]) if len(rec) > 1 else np.nan, 4),
        "malignant_precision":  round(float(prec[1]) if len(prec) > 1 else np.nan, 4),
        "benign_recall":        round(float(rec[0]) if len(rec) > 0 else np.nan, 4),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }

val_pat_metrics  = patient_level_metrics(val_pat_df,  t_pat, label="VAL")
test_pat_metrics = patient_level_metrics(test_pat_df, t_pat, label="TEST")

pat_report = pd.DataFrame([
    {"split": "val",  **val_pat_metrics},
    {"split": "test", **test_pat_metrics},
])
pat_report_path = TABLES / "patient_level_metrics_best_SET_A.csv"
pat_report.to_csv(pat_report_path, index=False)

print(f"  Best model: {best_name} | patient threshold (VAL): {t_pat:.3f}")
print()
print("  VAL  patient-level (reference only):")
for k in ["n_patients","accuracy","balanced_accuracy","mcc","macro_f1","roc_auc","pr_auc","brier","malignant_recall","tp","fp","tn","fn"]:
    print(f"    {k:<30} = {val_pat_metrics[k]}")
print()
print("  TEST patient-level (final report — no tuning on test):")
for k in ["n_patients","accuracy","balanced_accuracy","mcc","macro_f1","roc_auc","pr_auc","brier","malignant_recall","tp","fp","tn","fn"]:
    print(f"    {k:<30} = {test_pat_metrics[k]}")
print(f"\nSaved patient-level metrics: {pat_report_path}")
print()

Patient-level threshold (VAL, mean-agg): 0.140  [status=OK]

-- SECTION 16b: PATIENT-LEVEL TEST EVALUATION (final report — no tuning) ---
  Best model: Gradient Boosting | patient threshold (VAL): 0.140

  VAL  patient-level (reference only):
    n_patients                     = 123
    accuracy                       = 0.5285
    balanced_accuracy              = 0.6401
    mcc                            = 0.2134
    macro_f1                       = 0.4965
    roc_auc                        = 0.7227
    pr_auc                         = 0.4683
    brier                          = 0.1265
    malignant_recall               = 0.8095
    tp                             = 17
    fp                             = 54
    tn                             = 48
    fn                             = 4

  TEST patient-level (final report — no tuning on test):
    n_patients                     = 119
    accuracy                       = 0.5294
    balanced_accuracy              = 0.5539
    mcc           

---

## Section 18 — Save Best Model Bundle (joblib)

In [34]:
# =============================================================================
# 17) FUSION-READY CSV (SET-A, all models, all splits)
# =============================================================================
print("-- SECTION 10: EXPORT FUSION-READY CSV ------------------------------------")
frames = []
for split, X, y, img, pat in [("train", X_tr, y_tr, img_tr, pat_tr),
                                ("val",   X_v,  y_v,  img_v,  pat_v),
                                ("test",  X_te, y_te, img_te, pat_te)]:
    for name, model in trained_models.items():
        frames.append(make_img_df(model, X, y, img, pat, thresholds[name], name, split))
fusion_csv  = pd.concat(frames, ignore_index=True)
fusion_path = OUT / "fusion_probabilities_SET_A.csv"
fusion_csv.to_csv(fusion_path, index=False)
print(f"Saved fusion CSV: {fusion_path}  (rows={len(fusion_csv)})")
print()

-- SECTION 10: EXPORT FUSION-READY CSV ------------------------------------
Saved fusion CSV: /kaggle/working/metadata_classifier/fusion_probabilities_SET_A.csv  (rows=11202)



---

## Section 19 — Publication Figures (300 dpi) [UPDATED: ROC-AUC in threshold plot]

In [35]:
# =============================================================================
# 18) SAVE BEST MODEL BUNDLE
# =============================================================================
print("-- SECTION 11: SAVE BEST MODEL BUNDLE -------------------------------------")
bundle = {
    "model_object":            best_model,
    "best_model_name":         best_name,
    "threshold_image_level":   best_thr,
    "threshold_patient_level": t_pat,
    "feature_set":             "SET-A (clinical only — deployable)",
    "feature_names_raw":       FEATURES_A,
    "constraint": {
        "min_malignant_recall": TARGET_MALIGNANT_RECALL,
        "pos_rate_cap":         f"min({HARD_MAX_PRED_POS_RATE}, prevalence+{ADAPTIVE_POS_RATE_MARGIN})"
    },
    "cv_strategy":    "StratifiedGroupKFold(n_splits=5, groups=patient_id)",
    "calibration":    "OOF Platt scaling (LogisticRegression — public sklearn API)",
    "random_state":   RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "val_metrics_image_level": {
        k: float(res_val[best_name][k])
        for k in ["accuracy","balanced_accuracy","mcc","macro_f1",
                  "roc_auc","pr_auc","brier","malignant_recall","malignant_precision"]
    },
    "test_metrics_image_level_FINAL_REPORT_ONLY": {
        k: float(res_test[best_name][k])
        for k in ["accuracy","balanced_accuracy","mcc","macro_f1",
                  "roc_auc","pr_auc","brier","malignant_recall","malignant_precision"]
    },
    "val_patient_level_reference_only": {
        k: val_pat_metrics[k]
        for k in ["n_patients","accuracy","balanced_accuracy","mcc","macro_f1",
                  "roc_auc","pr_auc","brier","malignant_recall","malignant_precision"]
    },
    "test_patient_level_final_report_only": {
        k: test_pat_metrics[k]
        for k in ["n_patients","accuracy","balanced_accuracy","mcc","macro_f1",
                  "roc_auc","pr_auc","brier","malignant_recall","malignant_precision"]
    },
}
model_path = MODELS / "BEST_SET_A_model.joblib"
joblib.dump(bundle, model_path)
print(f"Saved: {model_path}")
print()
print("  Load example:")
print(f"""
  import joblib
  bundle = joblib.load(r"{model_path}")
  model  = bundle["model_object"]
  t_img  = bundle["threshold_image_level"]
  t_pat  = bundle["threshold_patient_level"]
  feats  = bundle["feature_names_raw"]
  p_mal  = model.predict_proba(X_new[feats])[:, 1]
  y_pred = (p_mal >= t_img).astype(int)
""")

-- SECTION 11: SAVE BEST MODEL BUNDLE -------------------------------------
Saved: /kaggle/working/metadata_classifier/saved_models/BEST_SET_A_model.joblib

  Load example:

  import joblib
  bundle = joblib.load(r"/kaggle/working/metadata_classifier/saved_models/BEST_SET_A_model.joblib")
  model  = bundle["model_object"]
  t_img  = bundle["threshold_image_level"]
  t_pat  = bundle["threshold_patient_level"]
  feats  = bundle["feature_names_raw"]
  p_mal  = model.predict_proba(X_new[feats])[:, 1]
  y_pred = (p_mal >= t_img).astype(int)



---

## Section 20 — Final Summary

In [36]:
# =============================================================================
# 19) PUBLICATION FIGURES (300 dpi)
# =============================================================================
print("-- SECTION 12: PUBLICATION FIGURES ----------------------------------------")
COLORS = plt.cm.tab10.colors

# Fig 1: ROC Curves (TEST)
fig, ax = plt.subplots(figsize=(7, 6))
for i, (name, m) in enumerate(res_test.items()):
    lw = 2.5 if name == best_name else 1.5
    ls = "-" if name == best_name else "--"
    label = f"{name}* (AUC={m['roc_auc']:.3f})" if name == best_name else f"{name} (AUC={m['roc_auc']:.3f})"
    ax.plot(m["fpr"], m["tpr"], color=COLORS[i], lw=lw, ls=ls, label=label)
ax.plot([0,1],[0,1], "k--", lw=1, label="Chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — TEST  (* = best model selected on VAL)")
ax.legend(fontsize=8, loc="lower right"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig1_roc_TEST.png", bbox_inches="tight")
plt.close()
print("  fig1_roc_TEST.png")

# Fig 2: PR Curves (TEST)
fig, ax = plt.subplots(figsize=(7, 6))
for i, (name, m) in enumerate(res_test.items()):
    lw = 2.5 if name == best_name else 1.5
    ls = "-" if name == best_name else "--"
    label = f"{name}* (AP={m['pr_auc']:.3f})" if name == best_name else f"{name} (AP={m['pr_auc']:.3f})"
    ax.plot(m["rcc"], m["prc"], color=COLORS[i], lw=lw, ls=ls, label=label)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — TEST  (* = best model selected on VAL)")
ax.legend(fontsize=8, loc="lower left"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig2_pr_TEST.png", bbox_inches="tight")
plt.close()
print("  fig2_pr_TEST.png")

# Fig 3: Confusion Matrices (TEST, all models)
n_models  = len(trained_models)
ncols     = min(3, n_models)
nrows     = int(np.ceil(n_models / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes_flat = np.array(axes).ravel()
for i, (name, m) in enumerate(res_test.items()):
    ax = axes_flat[i]
    sns.heatmap(m["cmn"], annot=True, fmt=".2%", cmap="Blues", ax=ax,
                xticklabels=["Benign","Malignant"],
                yticklabels=["Benign","Malignant"], cbar=True)
    star = " *" if name == best_name else ""
    ax.set_title(f"{name}{star}\n(TEST, threshold={thresholds[name]:.3f})", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for j in range(i+1, len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.suptitle("Confusion Matrices — TEST (* = best model selected on VAL)", fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGS / "fig3_confusion_matrices_TEST.png", bbox_inches="tight")
plt.close()
print("  fig3_confusion_matrices_TEST.png")

# Fig 4: Classification Report (TEST, best model)
bm_test = res_test[best_name]
cr_str  = classification_report(y_te, bm_test["yhat"],
                                 target_names=["Benign","Malignant"], zero_division=0)
cr_dict = classification_report(y_te, bm_test["yhat"],
                                 target_names=["Benign","Malignant"],
                                 output_dict=True, zero_division=0)
cr_df = pd.DataFrame(cr_dict).T.drop(columns=["support"], errors="ignore")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
tbl_data = [[idx] + [f"{v:.4f}" if isinstance(v, float) else str(v)
                      for v in row.tolist()]
             for idx, row in cr_df.iterrows()]
col_labels = ["Class"] + list(cr_df.columns)
tbl = ax.table(cellText=tbl_data, colLabels=col_labels,
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#2E75B6"); cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#E8F4FF")
ax.set_title(f"Classification Report — {best_name} *  (TEST, t={best_thr:.3f})",
             fontsize=11, pad=15)
plt.tight_layout()
plt.savefig(FIGS / "fig4_classification_report_TEST_best_model.png", bbox_inches="tight")
plt.close()
print("  fig4_classification_report_TEST_best_model.png")
print(f"\n  Classification Report ({best_name}, TEST):\n{cr_str}")

# Fig 5: Calibration Curve (TEST, best model)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(bm_test["prob_pred"], bm_test["prob_true"],
        marker="o", lw=2, label=f"{best_name} (Brier={bm_test['brier']:.4f})")
ax.plot([0,1],[0,1], "k--", lw=1, label="Perfect calibration")
ax.fill_between([0,1],[0,1],[0,0.5], alpha=0.05, color="red", label="Overconfident")
ax.fill_between([0,1],[0,1],[0.5,1], alpha=0.05, color="blue", label="Underconfident")
ax.set_xlabel("Mean Predicted Probability (Malignant)")
ax.set_ylabel("Fraction of Positives (True Malignant)")
ax.set_title(f"Calibration Curve — {best_name} *\nTEST  (model selected on VAL)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig5_calibration_TEST_best_model.png", bbox_inches="tight")
plt.close()
print("  fig5_calibration_TEST_best_model.png")

# Fig 6: Threshold Selection on VAL (best model)
thr_df = thr_tables[best_name].copy()
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thr_df["threshold"], thr_df["malignant_recall"],
        lw=2, label="Malignant Recall", color="#d62728")
ax.plot(thr_df["threshold"], thr_df["balanced_accuracy"],
        lw=2, label="Balanced Accuracy", color="#1f77b4")
ax.plot(thr_df["threshold"], thr_df["mcc"],
        lw=2, label="MCC", color="#2ca02c")
if "roc_auc" in thr_df.columns:
    ax.plot(thr_df["threshold"], thr_df["roc_auc"],
            lw=1.5, ls="-.", label="ROC-AUC", color="#9467bd")
ax.plot(thr_df["threshold"], thr_df["score"],
        lw=2, ls="--", label="Composite Score", color="#ff7f0e")
cap_val = thr_df["cap"].iloc[0]
ax.axhline(TARGET_MALIGNANT_RECALL, color="#d62728", ls=":", lw=1.5,
           label=f"Min recall gate ({TARGET_MALIGNANT_RECALL})")
ax.axvline(best_thr, color="black", ls="--", lw=1.5,
           label=f"Selected t={best_thr:.3f}")
feasible_mask = (
    (thr_df["malignant_recall"] >= TARGET_MALIGNANT_RECALL) &
    (thr_df["pos_rate"] <= cap_val)
)
ax.fill_between(thr_df["threshold"],
                TARGET_MALIGNANT_RECALL, 1.0,
                where=feasible_mask,
                alpha=0.10, color="green", label="Feasible region")
ax.set_xlabel("Decision Threshold"); ax.set_ylabel("Score / Rate")
ax.set_title(f"Threshold Selection on VALIDATION — {best_name} *\n"
             f"Recall gate >= {TARGET_MALIGNANT_RECALL} | Selection: best composite score | "
             f"pos_rate cap={cap_val:.2f}")
ax.legend(fontsize=8, loc="lower left"); ax.grid(True, alpha=0.3)
ax.set_xlim(0.05, 0.95); ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(FIGS / "fig6_threshold_selection_VAL_best_model.png", bbox_inches="tight")
plt.close()
print("  fig6_threshold_selection_VAL_best_model.png")
print()

-- SECTION 12: PUBLICATION FIGURES ----------------------------------------
  fig1_roc_TEST.png
  fig2_pr_TEST.png
  fig3_confusion_matrices_TEST.png
  fig4_classification_report_TEST_best_model.png

  Classification Report (Gradient Boosting, TEST):
              precision    recall  f1-score   support

      Benign       0.88      0.66      0.75       239
   Malignant       0.24      0.53      0.33        47

    accuracy                           0.64       286
   macro avg       0.56      0.60      0.54       286
weighted avg       0.77      0.64      0.68       286

  fig5_calibration_TEST_best_model.png
  fig6_threshold_selection_VAL_best_model.png



---

## Section 21

In [37]:
# =============================================================================
# 20) FINAL SUMMARY
# =============================================================================
print("=" * 80)
print("FINAL SUMMARY (Q1-SAFE)")
print("=" * 80)
print(f"""
Best deployable model (VAL selection):  {best_name}
Image-level threshold (VAL):            {best_thr:.3f}
Patient-level threshold (VAL mean-agg): {t_pat:.3f}

VAL image-level metrics:
  BalancedAcc = {res_val[best_name]['balanced_accuracy']:.4f}
  ROC-AUC     = {res_val[best_name]['roc_auc']:.4f}
  MCC         = {res_val[best_name]['mcc']:.4f}
  MalRecall   = {res_val[best_name]['malignant_recall']:.4f}

TEST image-level metrics (final report — not used for selection):
  BalancedAcc = {res_test[best_name]['balanced_accuracy']:.4f}
  ROC-AUC     = {res_test[best_name]['roc_auc']:.4f}
  MCC         = {res_test[best_name]['mcc']:.4f}
  MalRecall   = {res_test[best_name]['malignant_recall']:.4f}

Outputs:
  Model bundle : {model_path}
  Fusion CSV   : {fusion_path}
  Tables       : {TABLES}
  Figures      : {FIGS}
""")
print("DONE")

FINAL SUMMARY (Q1-SAFE)

Best deployable model (VAL selection):  Gradient Boosting
Image-level threshold (VAL):            0.160
Patient-level threshold (VAL mean-agg): 0.140

VAL image-level metrics:
  BalancedAcc = 0.7051
  ROC-AUC     = 0.7450
  MCC         = 0.3303
  MalRecall   = 0.8000

TEST image-level metrics (final report — not used for selection):
  BalancedAcc = 0.5965
  ROC-AUC     = 0.6439
  MCC         = 0.1481
  MalRecall   = 0.5319

Outputs:
  Model bundle : /kaggle/working/metadata_classifier/saved_models/BEST_SET_A_model.joblib
  Fusion CSV   : /kaggle/working/metadata_classifier/fusion_probabilities_SET_A.csv
  Tables       : /kaggle/working/metadata_classifier/tables
  Figures      : /kaggle/working/metadata_classifier/figures

DONE


In [38]:
import shutil

source_folder = "/kaggle/working/"
zip_path = "/kaggle/working/metadata_final"

shutil.make_archive(
    base_name=zip_path.replace(".zip",""),
    format="zip",
    root_dir=source_folder
)

print("✅ Zipped successfully:", zip_path)


✅ Zipped successfully: /kaggle/working/metadata_final
